In [20]:
import pandas as pd

# 1. Load and preprocess data
parent = pd.read_csv('parent_order.csv')
child = pd.read_csv('child_order.csv')
trade = pd.read_csv('trade.csv')
quote = pd.read_csv('quote.csv')

# Combine date and time into datetime and sort to ensure accurate merge_asof
for df in [child, trade, quote]:
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
    df.sort_values('datetime', inplace=True)

results = []

for _, p_order in parent.iterrows():
    order_id, sym, date, side = p_order['orderid'], p_order['sym'], p_order['date'], p_order['side']

    # Order start and end times
    start_time = pd.to_datetime(f"{date} {p_order['starttime']}")
    end_time = pd.to_datetime(f"{date} {p_order['endtime']}")

    # Filter data slices for the current order
    c_order = child[child['parentid'] == order_id]
    m_trade = trade[(trade['sym'] == sym) & (trade['date'] == date)].copy()
    m_quote = quote[(quote['sym'] == sym) & (quote['date'] == date)]

    # Handle late trading delays: normalize trades after 14:57 to 15:00:00 and resort
    m_trade.loc[m_trade['datetime'].dt.time >= pd.to_datetime('14:57:00').time(), 'datetime'] = pd.to_datetime(f"{date} 15:00:00")
    m_trade.sort_values('datetime', inplace=True)

    # ---------------- Core Metrics Calculation ----------------
    total_qty = c_order['size'].sum()
    notional = (c_order['price'] * c_order['size']).sum()
    avg_price = notional / total_qty

    adv_pct = total_qty / m_trade['size'].sum()

    interval_quotes = m_quote[(m_quote['datetime'] >= start_time) & (m_quote['datetime'] <= end_time)]
    spread_bps = ((interval_quotes['ask'] - interval_quotes['bid']) / ((interval_quotes['ask'] + interval_quotes['bid']) / 2) * 10000).mean()

    # ---------------- Benchmark Prices ----------------
    open_price = m_trade.iloc[0]['price']
    close_price = m_trade.iloc[-1]['price']

    # Arrival Price: Use open price if before market open, otherwise use the latest mid-price before order placement
    if start_time.time() <= pd.to_datetime('09:30:00').time():
        arrival_price = open_price
    else:
        arrival_q = m_quote[m_quote['datetime'] <= start_time]
        arrival_price = (arrival_q.iloc[-1]['bid'] + arrival_q.iloc[-1]['ask']) / 2 if not arrival_q.empty else open_price

    # IVWAP
    interval_trades = m_trade[(m_trade['datetime'] >= start_time) & (m_trade['datetime'] <= end_time)]
    ivwap_price = (interval_trades['price'] * interval_trades['size']).sum() / interval_trades['size'].sum()

    # PWPS (Simulate execution at a 5% participation rate)
    pwp_trades = m_trade[m_trade['datetime'] >= start_time].copy()
    pwp_trades['sim_qty'] = pwp_trades['size'] * 0.05
    pwp_trades['cum_sim_qty'] = pwp_trades['sim_qty'].cumsum()

    overshoot_idx = pwp_trades[pwp_trades['cum_sim_qty'] > total_qty].index
    if not overshoot_idx.empty:
        first_overshoot = overshoot_idx[0]
        # Truncate the excess portion so the sum exactly matches total_qty
        pwp_trades.loc[first_overshoot, 'sim_qty'] -= (pwp_trades.loc[first_overshoot, 'cum_sim_qty'] - total_qty)
        pwp_trades = pwp_trades.loc[:first_overshoot]

    pwps_price = (pwp_trades['price'] * pwp_trades['sim_qty']).sum() / total_qty

    # ---------------- Fill Statistics and Cost Calculation ----------------
    calc_cost = lambda bench_p: side * (avg_price - bench_p) / bench_p * 10000

    moo_pct = c_order[c_order['datetime'].dt.time == pd.to_datetime('09:25:00').time()]['size'].sum() / total_qty
    moc_pct = c_order[c_order['datetime'].dt.time >= pd.to_datetime('14:57:00').time()]['size'].sum() / total_qty

    # Match the latest quote to calculate aggressive/passive fills
    c_order_q = pd.merge_asof(c_order, m_quote[['datetime', 'bid', 'ask']], on='datetime', direction='backward')
    passive_qty = aggressive_qty = 0

    for _, row in c_order_q.iterrows():
        p, b, a, sz = row['price'], row['bid'], row['ask'], row['size']
        if side == 1:
            if p <= b: passive_qty += sz
            elif p >= a: aggressive_qty += sz
        else:
            if p >= a: passive_qty += sz
            elif p <= b: aggressive_qty += sz

    results.append({
        'OrderID': order_id,
        'Notional (Million CNY)': notional / 1_000_000,
        'Trading Speed (ADV%)': adv_pct,
        'Spread (bps)': spread_bps,
        'Open': calc_cost(open_price),
        'Arrival': calc_cost(arrival_price),
        'IVWAP': calc_cost(ivwap_price),
        'Close': calc_cost(close_price),
        'PWPS': calc_cost(pwps_price),
        'MOO%': moo_pct,
        'MOC%': moc_pct,
        'Passive': passive_qty / total_qty,
        'Aggressive': aggressive_qty / total_qty
    })

# 2. Format and export results
df = pd.DataFrame(results)

for col in ['Trading Speed (ADV%)', 'MOO%', 'MOC%', 'Passive', 'Aggressive']:
    df[col] = df[col].apply(lambda x: f"{x * 100:.2f}%")
for col in ['Notional (Million CNY)', 'Spread (bps)', 'Open', 'Arrival', 'IVWAP', 'Close', 'PWPS']:
    df[col] = df[col].apply(lambda x: f"{x:.2f}")

df.to_csv('Assignment_2_Output.csv', index=False)